# 새 Parser 확인: 결과가 바뀌는 53장만 실제로 돌리기

예선 최종본의 OCR·cascade(`ocr_pipeline.py`)는 그대로 두고, **날짜 해석(Parser)만 새 버전(`parser_v2/date_parser`)으로 바꿔서**
실제 제출 함수 `predict_image`로 53장을 돌립니다.

- 대상: 저장된 OCR로 미리 계산했을 때 결과가 바뀌는 53장 (새로 맞히는 41장 + 처리 단계만 바뀌는 12장)
- 비교: ① 예선 최종본 실제 결과(`run701_output/predictions.csv`) ② 미리 계산한 예상 결과
- 예선 최종본 파일(`date_parser`, `ocr_pipeline.py`)은 **수정하지 않습니다**. 새 Parser는 `parser_v2` 폴더에서만 불러옵니다.
- 시간: 약 15~25분. 중간에 멈춰도 다시 "모두 실행"하면 이어서 진행합니다.

**반드시 커널을 새로 시작한 뒤(Kernel → Restart) 모두 실행하세요.** 이전 노트북에서 불러온 예선 Parser가 남아 있으면 새 Parser가 적용되지 않습니다.

In [ ]:
# ===== CONFIG (기본값 그대로 두면 됩니다) =====
from pathlib import Path
REPO_DIR       = Path(r"C:\Users\user\itda_OCR")                     # 예선 최종본 폴더 (수정하지 않음)
NEW_PARSER_DIR = Path(r"C:\Users\user\itda_OCR\parser_v2")           # 새 Parser (zip에서 풀어 넣은 폴더)
IMAGE_DIR      = Path(r"C:\Users\user\itda_OCR\상품사진입니다")
LABELS_CSV     = Path(r"C:\Users\user\itda_OCR\labels_701_v2.csv")   # 라벨 7장 수정 반영본
OLD_PRED_CSV   = Path(r"C:\Users\user\itda_OCR\run701_output\predictions.csv")  # 예선 최종본 실제 결과
OUTPUT_DIR     = Path(r"C:\Users\user\itda_OCR\parser_v2_check")
CPU_THREADS    = 4
# ============================================

In [ ]:
# 1) 새 Parser를 먼저 불러오도록 경로 설정 + 확인
import sys, os, json, csv, time
for p, name in [(REPO_DIR / "ocr_pipeline.py", "REPO_DIR"), (NEW_PARSER_DIR / "date_parser" / "hints.py", "NEW_PARSER_DIR"),
                (IMAGE_DIR, "IMAGE_DIR"), (LABELS_CSV, "LABELS_CSV"), (OLD_PRED_CSV, "OLD_PRED_CSV")]:
    if not p.exists():
        raise FileNotFoundError(f"{name} 확인 필요: {p} 가 없습니다.")
if "date_parser" in sys.modules:
    raise RuntimeError("이전 Parser가 이미 불러와져 있습니다. 메뉴에서 Kernel → Restart 후 다시 모두 실행하세요.")
sys.path.insert(0, str(NEW_PARSER_DIR))
sys.path.insert(1, str(REPO_DIR))
os.chdir(REPO_DIR)

import date_parser, ocr_pipeline as op
parser_path = Path(date_parser.__file__).resolve().parent
if parser_path != (NEW_PARSER_DIR / "date_parser").resolve():
    raise RuntimeError(f"새 Parser가 아니라 {parser_path} 를 불러왔습니다. 커널을 재시작하세요.")
print("사용 중인 Parser:", parser_path)
print("사용 중인 OCR/cascade:", Path(op.__file__).resolve())
print("새 Parser 버전:", (NEW_PARSER_DIR / "VERSION.txt").read_text(encoding="utf-8").strip() if (NEW_PARSER_DIR / "VERSION.txt").exists() else "?")

In [ ]:
# 2) 대상 53장과 미리 계산한 예상 결과
CASES = {
"36": {
"old": "2026-08-26",
"expected_new": "2026-08-26",
"expected_method": "original_512"
},
"198": {
"old": "NONE",
"expected_new": "2026-05-08",
"expected_method": "clahe"
},
"384": {
"old": "2025-07-NONE",
"expected_new": "2026-07-08",
"expected_method": "original_512"
},
"406": {
"old": "NONE",
"expected_new": "2026-05-27",
"expected_method": "highres_1024"
},
"432": {
"old": "2027-04-28",
"expected_new": "2027-04-28",
"expected_method": "original_512"
},
"515": {
"old": "2022-09-22",
"expected_new": "2026-01-NONE",
"expected_method": "original_512"
},
"682": {
"old": "NONE",
"expected_new": "2026-01-25",
"expected_method": "original_512"
},
"686": {
"old": "2025-10-NONE",
"expected_new": "2025-10-12",
"expected_method": "original_512"
},
"715": {
"old": "NONE",
"expected_new": "2026-06-26",
"expected_method": "original_512"
},
"789": {
"old": "NONE",
"expected_new": "NONE-12-19",
"expected_method": "original_512"
},
"824": {
"old": "2018-12-10",
"expected_new": "NONE-12-18",
"expected_method": "original_512"
},
"832": {
"old": "2025-07-NONE",
"expected_new": "2025-07-15",
"expected_method": "clahe"
},
"892": {
"old": "NONE",
"expected_new": "2022-02-07",
"expected_method": "original_512"
},
"893": {
"old": "NONE",
"expected_new": "2022-02-07",
"expected_method": "original_512"
},
"913": {
"old": "NONE",
"expected_new": "2021-10-19",
"expected_method": "rotation_270"
},
"977": {
"old": "2020-11-03",
"expected_new": "2021-11-03",
"expected_method": "original_512"
},
"1070": {
"old": "NONE",
"expected_new": "NONE-01-24",
"expected_method": "original_512"
},
"1071": {
"old": "NONE",
"expected_new": "NONE-01-24",
"expected_method": "original_512"
},
"1131": {
"old": "NONE",
"expected_new": "NONE-02-12",
"expected_method": "original_512"
},
"1205": {
"old": "NONE",
"expected_new": "2021-05-NONE",
"expected_method": "highres_1024"
},
"1388": {
"old": "2021-07-NONE",
"expected_new": "2021-07-17",
"expected_method": "original_512"
},
"1554": {
"old": "NONE",
"expected_new": "2020-06-03",
"expected_method": "rotation_270"
},
"1730": {
"old": "2021-10-28",
"expected_new": "2021-10-02",
"expected_method": "original_512"
},
"1733": {
"old": "2022-11-18",
"expected_new": "2022-11-18",
"expected_method": "original_512"
},
"1741": {
"old": "2022-11-13",
"expected_new": "2022-11-01",
"expected_method": "original_512"
},
"1757": {
"old": "NONE",
"expected_new": "2023-03-09",
"expected_method": "original_512"
},
"1793": {
"old": "2021-11-07",
"expected_new": "2021-11-07",
"expected_method": "original_512"
},
"1851": {
"old": "NONE",
"expected_new": "2021-12-16",
"expected_method": "original_512"
},
"1855": {
"old": "NONE",
"expected_new": "NONE-04-10",
"expected_method": "highres_1024"
},
"1857": {
"old": "2022-07-17",
"expected_new": "2022-07-17",
"expected_method": "original_512"
},
"1912": {
"old": "NONE",
"expected_new": "2021-05-21",
"expected_method": "original_512"
},
"2011": {
"old": "NONE",
"expected_new": "2020-06-30",
"expected_method": "original_512"
},
"2012": {
"old": "NONE",
"expected_new": "2020-06-30",
"expected_method": "original_512"
},
"2048": {
"old": "NONE",
"expected_new": "2021-07-31",
"expected_method": "original_512"
},
"2051": {
"old": "NONE",
"expected_new": "NONE-04-14",
"expected_method": "original_512"
},
"2148": {
"old": "NONE",
"expected_new": "2022-01-NONE",
"expected_method": "highres_1024"
},
"2159": {
"old": "NONE",
"expected_new": "NONE-04-13",
"expected_method": "original_512"
},
"2220": {
"old": "2019-09-30",
"expected_new": "2019-09-30",
"expected_method": "rotation_270"
},
"2247": {
"old": "2020-09-21",
"expected_new": "2021-09-07",
"expected_method": "original_512"
},
"2251": {
"old": "NONE",
"expected_new": "2021-05-09",
"expected_method": "highres_1024"
},
"2258": {
"old": "2022-01-24",
"expected_new": "2022-01-24",
"expected_method": "original_512"
},
"2265": {
"old": "NONE",
"expected_new": "NONE-03-05",
"expected_method": "original_512"
},
"2310": {
"old": "NONE",
"expected_new": "NONE-03-13",
"expected_method": "original_512"
},
"2345": {
"old": "NONE",
"expected_new": "NONE-02-12",
"expected_method": "original_512"
},
"2421": {
"old": "NONE",
"expected_new": "2020-10-23",
"expected_method": "original_512"
},
"2649": {
"old": "2021-07-NONE",
"expected_new": "2021-07-01",
"expected_method": "original_512"
},
"2731": {
"old": "NONE",
"expected_new": "2021-04-18",
"expected_method": "original_512"
},
"2835": {
"old": "2021-06-NONE",
"expected_new": "2021-06-30",
"expected_method": "original_512"
},
"2847": {
"old": "2021-12-NONE",
"expected_new": "2021-12-30",
"expected_method": "original_512"
},
"2927": {
"old": "NONE",
"expected_new": "NONE-04-08",
"expected_method": "rotation_270"
},
"3016": {
"old": "NONE",
"expected_new": "2022-07-NONE",
"expected_method": "original_512"
},
"3203": {
"old": "NONE",
"expected_new": "2021-10-NONE",
"expected_method": "clahe"
},
"3277": {
"old": "NONE",
"expected_new": "NONE-04-05",
"expected_method": "highres_1024"
}
}

truth = {str(int(r["image_id"])): r["final_date"] for r in csv.DictReader(open(LABELS_CSV, encoding="utf-8-sig"))}
old_pred = {str(int(r["image_id"])): r["final_date"] for r in csv.DictReader(open(OLD_PRED_CSV, encoding="utf-8-sig"))}
files = {}
for p in IMAGE_DIR.rglob("*"):
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"} and p.stem.isdigit():
        files.setdefault(str(int(p.stem)), p)
missing = [i for i in CASES if i not in files]
print("대상:", len(CASES), "장 / 못 찾은 이미지:", missing or "없음")

In [ ]:
# 3) OCR 엔진 준비 (예선 최종본 설정 그대로)
engine = op.initialize_engine(enable_mkldnn=True, cpu_threads=CPU_THREADS, recognition_batch_size=6)
print("OCR 엔진 준비 완료")

In [ ]:
# 4) 실행: 예선 최종본의 predict_image 그대로 (Parser만 새 버전)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULT = OUTPUT_DIR / "results.jsonl"
done = {}
if RESULT.exists():
    for line in open(RESULT, encoding="utf-8"):
        try:
            r = json.loads(line); done[r["image_id"]] = r
        except json.JSONDecodeError:
            pass
todo = [i for i in CASES if i in files and i not in done]
print(f"이미 완료 {len(done)}장, 이번에 실행 {len(todo)}장")
started = time.perf_counter()
with open(RESULT, "a", encoding="utf-8") as out:
    for n, i in enumerate(todo, 1):
        t0 = time.perf_counter()
        prediction, method = op.predict_image(engine, files[i])
        rec = {"image_id": i, "pred": prediction["final_date"], "method": method, "seconds": round(time.perf_counter() - t0, 1)}
        out.write(json.dumps(rec, ensure_ascii=False) + "\n"); out.flush()
        done[i] = rec
        if n % 5 == 0 or n == len(todo):
            el = time.perf_counter() - started
            print(f"[{n}/{len(todo)}] 경과 {el/60:.1f}분, 남은 예상 {el/n*(len(todo)-n)/60:.1f}분", flush=True)
print("실행 완료")

In [ ]:
# 5) 비교
rows = []
for i in CASES:
    if i not in done:
        continue
    new, old, t = done[i]["pred"], old_pred.get(i), truth[i]
    rows.append({"image_id": i, "truth": t, "old(예선)": old, "new(새 Parser)": new, "expected(미리 계산)": CASES[i]["expected_new"],
                 "old_ok": old == t, "new_ok": new == t, "matches_expected": new == CASES[i]["expected_new"],
                 "method": done[i]["method"], "seconds": done[i]["seconds"]})
with open(OUTPUT_DIR / "comparison.csv", "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

gained = [r for r in rows if r["new_ok"] and not r["old_ok"]]
lost = [r for r in rows if r["old_ok"] and not r["new_ok"]]
off = [r for r in rows if not r["matches_expected"]]
print(f"실행 {len(rows)}장 | 예선 정답 {sum(r['old_ok'] for r in rows)}장 → 새 Parser 정답 {sum(r['new_ok'] for r in rows)}장")
print(f"새로 맞힘 {len(gained)}장 / 새로 틀림 {len(lost)}장 / 미리 계산과 다른 결과 {len(off)}장")
for r in lost + off:
    print("  확인 필요:", r)
print("\nClaude에게 보낼 파일:", OUTPUT_DIR / "comparison.csv")